# Stage 5 – Dataset & Input Preprocessing Experiments

In [20]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.npy'):
            print(os.path.join(root, file))

/kaggle/input/notebooks/omarkhaled67/notebooka4d2af853e/y_val.npy
/kaggle/input/notebooks/omarkhaled67/notebooka4d2af853e/y_train_augmented.npy
/kaggle/input/notebooks/omarkhaled67/notebooka4d2af853e/X_val.npy
/kaggle/input/notebooks/omarkhaled67/notebooka4d2af853e/X_train_augmented.npy
/kaggle/input/datasets/omarashour206/aptos2019-preprocessed-dataset/y_train.npy
/kaggle/input/datasets/omarashour206/aptos2019-preprocessed-dataset/y_val.npy
/kaggle/input/datasets/omarashour206/aptos2019-preprocessed-dataset/X_val.npy
/kaggle/input/datasets/omarashour206/aptos2019-preprocessed-dataset/X_train.npy


In [21]:
import numpy as np

DATA_PATH = '/kaggle/input/notebooks/omarkhaled67/notebooka4d2af853e/'

X_train = np.load(DATA_PATH + 'X_train_augmented.npy')
y_train = np.load(DATA_PATH + 'y_train_augmented.npy')

X_val = np.load(DATA_PATH + 'X_val.npy')
y_val = np.load(DATA_PATH + 'y_val.npy')

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("Pixel range:", X_train.min(), X_train.max())
print("Classes:", np.unique(y_train))

X_train: (4489, 224, 224, 3)
y_train: (4489,)
X_val: (733, 224, 224, 3)
y_val: (733,)
Pixel range: 0.0 1.0
Classes: [0 1 2 3 4]


In [22]:
import os

MODEL_PATH = '/kaggle/working/best_efficientnet_b0_exp50.keras'

print("Model exists:", os.path.exists(MODEL_PATH))

Model exists: True


In [23]:
import tensorflow as tf
import numpy as np

MODEL_PATH = '/kaggle/working/best_efficientnet_b0_exp50.keras'

model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully")
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

Model loaded successfully
Train: (4489, 224, 224, 3) (4489,)
Validation: (733, 224, 224, 3) (733,)


In [24]:
print("Optimizer:", model.optimizer)
print("Learning rate:", model.optimizer.learning_rate.numpy())
print("Output shape:", model.output_shape)

Optimizer: <keras.src.optimizers.adam.Adam object at 0x7db458635d10>
Learning rate: 3.125e-07
Output shape: (None, 5)


In [25]:

import tensorflow as tf
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

                                   
base_model = None

for layer in model.layers:
    if isinstance(layer, tf.keras.Model) and "efficientnet" in layer.name.lower():
        base_model = layer
        break

if base_model is None:
    raise ValueError(
        "EfficientNet backbone was not found. "
        "Print model.summary() before continuing."
    )

                                 
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

                                        
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

                                          
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

                             
checkpoint_path = (
    "/kaggle/working/best_efficientnet_b0_new_preprocessing.keras"
)

callbacks = [
    ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

                             
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=12,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/12


2026-10-13 20:06:25.684310: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-13 20:06:25.824368: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-13 20:06:26.162962: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-13 20:06:26.307014: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-13 20:06:26.450627: E external/local_xla/xla/stream_

281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.2368 - loss: 1.8226

2026-10-13 20:07:09.088590: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-13 20:07:09.227325: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-13 20:07:09.562924: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-13 20:07:09.706275: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-13 20:07:10.491944: E external/local_xla/xla/stream_


Epoch 1: val_loss improved from None to 1.47534, saving model to /kaggle/working/best_efficientnet_b0_new_preprocessing.keras

Epoch 1: finished saving model to /kaggle/working/best_efficientnet_b0_new_preprocessing.keras
281/281 ━━━━━━━━━━━━━━━━━━━━ 67s 138ms/step - accuracy: 0.2767 - loss: 1.6477 - val_accuracy: 0.4925 - val_loss: 1.4753 - learning_rate: 5.0000e-06
Epoch 2/12
280/281 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2876 - loss: 1.5342
Epoch 2: val_loss did not improve from 1.47534
281/281 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.2894 - loss: 1.5349 - val_accuracy: 0.4925 - val_loss: 1.5251 - learning_rate: 5.0000e-06
Epoch 3/12
279/281 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3015 - loss: 1.5468
Epoch 3: val_loss did not improve from 1.47534

Epoch 3: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-06.
281/281 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.2976 - loss: 1.5314 - val_accuracy: 0.4925 - val_loss: 1.5028 - learning_rate: 5.0

In [26]:
import tensorflow as tf

old_model = tf.keras.models.load_model(
    '/kaggle/working/best_efficientnet_b0_exp50.keras'
)

results = old_model.evaluate(
    X_val,
    y_val,
    verbose=1
)

print("Old model validation loss:", results[0])
print("Old model validation accuracy:", results[1])

23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 233ms/step - accuracy: 0.4925 - loss: 1.4643
Old model validation loss: 1.4642658233642578
Old model validation accuracy: 0.4924965798854828


In [27]:
import numpy as np

OLD_DATA_PATH = '/kaggle/input/datasets/omarashour206/aptos2019-preprocessed-dataset/'

X_train = np.load(OLD_DATA_PATH + 'X_train.npy')
y_train = np.load(OLD_DATA_PATH + 'y_train.npy')

X_val = np.load(OLD_DATA_PATH + 'X_val.npy')
y_val = np.load(OLD_DATA_PATH + 'y_val.npy')

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

Train: (4489, 224, 224, 3) (4489,)
Validation: (733, 224, 224, 3) (733,)


In [28]:
import tensorflow as tf

model = tf.keras.models.load_model(
    '/kaggle/working/best_efficientnet_b0_exp50.keras'
)

results = model.evaluate(X_val, y_val, verbose=1)

print(f"Validation Accuracy: {results[1] * 100:.2f}%")

23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 233ms/step - accuracy: 0.4925 - loss: 1.4643
Validation Accuracy: 49.25%


In [29]:
import numpy as np

OLD_PATH = '/kaggle/input/datasets/omarashour206/aptos2019-preprocessed-dataset/'
NEW_PATH = '/kaggle/input/notebooks/omarkhaled67/notebooka4d2af853e/'

for name, path in [
    ("Old", OLD_PATH),
    ("New", NEW_PATH)
]:
    print(f"\n{name} dataset")

    for file in ["X_val.npy", "y_val.npy"]:
        arr = np.load(path + file, mmap_mode="r")
        print(file, arr.shape, arr.dtype)


Old dataset
X_val.npy (733, 224, 224, 3) float32
y_val.npy (733,) int64

New dataset
X_val.npy (733, 224, 224, 3) float32
y_val.npy (733,) int64


In [30]:
import numpy as np

OLD_PATH = '/kaggle/input/datasets/omarashour206/aptos2019-preprocessed-dataset/'
NEW_PATH = '/kaggle/input/notebooks/omarkhaled67/notebooka4d2af853e/'

X_val_old = np.load(OLD_PATH + 'X_val.npy')
y_val_old = np.load(OLD_PATH + 'y_val.npy')

X_val_new = np.load(NEW_PATH + 'X_val.npy')
y_val_new = np.load(NEW_PATH + 'y_val.npy')

print("Same labels:", np.array_equal(y_val_old, y_val_new))
print("Same images:", np.allclose(X_val_old, X_val_new, atol=1e-6))
print("Old pixel range:", X_val_old.min(), X_val_old.max())
print("New pixel range:", X_val_new.min(), X_val_new.max())

Same labels: True
Same images: False
Old pixel range: 0.0 1.0
New pixel range: 0.0 1.0


In [31]:
import tensorflow as tf
import numpy as np

MODEL_PATH = '/kaggle/working/best_efficientnet_b0_exp50.keras'

model = tf.keras.models.load_model(MODEL_PATH)

                                
loss_01, acc_01 = model.evaluate(X_val_old, y_val_old, verbose=0)

                                  
loss_255, acc_255 = model.evaluate(X_val_old * 255.0, y_val_old, verbose=0)

print(f"Accuracy with [0, 1]:   {acc_01 * 100:.2f}%")
print(f"Accuracy with [0, 255]: {acc_255 * 100:.2f}%")

Accuracy with [0, 1]:   49.25%
Accuracy with [0, 255]: 77.76%


In [32]:
val_loss, val_accuracy = model.evaluate(
    X_val_old * 255.0,
    y_val_old,
    verbose=1
)

print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.7776 - loss: 0.6034
Validation Accuracy: 77.76%


In [33]:
X_train_scaled = X_train * 255.0
X_val_scaled = X_val * 255.0

In [34]:
import numpy as np
import tensorflow as tf

OLD_DATA_PATH = '/kaggle/input/datasets/omarashour206/aptos2019-preprocessed-dataset/'

X_train = np.load(OLD_DATA_PATH + 'X_train.npy')
y_train = np.load(OLD_DATA_PATH + 'y_train.npy')

X_val = np.load(OLD_DATA_PATH + 'X_val.npy')
y_val = np.load(OLD_DATA_PATH + 'y_val.npy')

                                              
X_train = X_train * 255.0
X_val = X_val * 255.0

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Train pixel range:", X_train.min(), X_train.max())
print("Validation pixel range:", X_val.min(), X_val.max())

X_train: (4489, 224, 224, 3)
X_val: (733, 224, 224, 3)
Train pixel range: 0.0 255.0
Validation pixel range: 0.0 255.0
